# Step 09.1 — Backpropagation Through Time：把 RNN 的 Chain Rule 沿时间展开

Step 8 已经解决 forward recurrence：

$$h_t=\tanh(W_xx_t+W_hh_{t-1}+b).$$

现在进入训练最关键的问题：如果较晚的 prediction 产生 loss，早期状态和被反复使用的 RNN parameters 怎样收到 gradient？

本节五部分：① 把 recurrent graph 沿时间展开；② 推导 hidden-state gradient 为什么包含未来路径；③ 用 3-step scalar RNN 手算完整 BPTT；④ 解释 shared parameter 的 gradient 为什么是各时间步贡献之和，并用 MLX autodiff 验证；⑤ 看见 long-range chain 中的导数乘积，为 9.2 vanishing/exploding gradients 做准备。

这节只研究 BPTT 结构。梯度为什么会消失/爆炸的规模分析留到 Step 9.2。


## 1. “Through Time” 是什么？——同一个 Cell 在计算图上被展开

RNN forward：

```text
h_(-1) --[Cell at t=0]--> h_0 --[Cell at t=1]--> h_1 --[Cell at t=2]--> h_2
              ↑ x_0                  ↑ x_1                  ↑ x_2
```

这里画了三个 Cell 节点，但不是三个不同模型。三个节点全部引用同一个 `W_x / W_h / b`。

训练时，autodiff 看到的是一张沿时间变长的 computation graph。BPTT 的核心就是：先把这条 recurrent dependency 看成普通的展开图，再从 loss 沿所有依赖路径反向应用 chain rule。

$$\boxed{\text{BPTT = backpropagation on an unrolled recurrent computation graph}}$$

它不是另发明一套导数规则。


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.set_xlim(-0.7, 9.7)
ax.set_ylim(-1.0, 2.0)
ax.axis('off')

xs_plot = [0.5, 3.5, 6.5]
for t, xpos in enumerate(xs_plot):
    ax.text(xpos, 0.6, f'h_{t}', ha='center', va='center',
            bbox={'boxstyle':'round,pad=0.35', 'fill':False})
    ax.text(xpos, 1.55, f'x_{t}', ha='center', va='center')
    ax.annotate('', xy=(xpos, 0.95), xytext=(xpos, 1.35),
                arrowprops={'arrowstyle':'->'})
    if t > 0:
        ax.annotate('', xy=(xpos-0.45, 0.6), xytext=(xs_plot[t-1]+0.45, 0.6),
                    arrowprops={'arrowstyle':'->'})

ax.text(9.0, 0.6, 'Loss', ha='center', va='center',
        bbox={'boxstyle':'round,pad=0.35', 'fill':False})
ax.annotate('', xy=(8.45,0.6), xytext=(7.0,0.6),
            arrowprops={'arrowstyle':'->'})
ax.text(3.5, -0.45, 'same W_x, W_h, b are reused at every time step',
        ha='center', va='center')
ax.set_title('RNN unrolled through time')
plt.show()


计算图“展开”只是分析视角：代码仍然可以是一个 `for t in range(T)`。我们不是复制 parameter values，而是让同一 parameters 在图中参与多次运算。

因为同一个 parameter 出现在多个路径里，reverse-mode autodiff 最终必须把这些路径对它的 gradient contribution 累加起来。


## 2. Hidden-State Gradient：当前时刻既受本地 Loss，也受未来影响

假设每个时间步都可以有一个 local loss：

$$L=\ell_0+\ell_1+\cdots+\ell_{T-1}.$$

中间状态 `h_t` 可以通过两条主要路径影响总 loss：当前 local loss，以及 `h_(t+1)` 之后的未来 losses。

因此在简单 recurrent chain 上：

$$
\boxed{
\frac{\partial L}{\partial h_t}
=
\frac{\partial \ell_t}{\partial h_t}
+
\frac{\partial L}{\partial h_{t+1}}
\frac{\partial h_{t+1}}{\partial h_t}
}
$$

第二项中的 `∂L/∂h_(t+1)` 已经包含更远未来的影响，因此会继续向早期传播。

### Scalar Vanilla RNN

$$a_t=w_xx_t+w_hh_{t-1}+b,$$

$$h_t=\tanh(a_t).$$

因为

$$\frac{d\tanh(a_t)}{da_t}=1-h_t^2,$$

以及

$$\frac{\partial a_t}{\partial h_{t-1}}=w_h,$$

所以

$$
\boxed{
\frac{\partial h_t}{\partial h_{t-1}}
=
w_h(1-h_t^2)
}
$$

这一个局部因子以后会在很多时间步之间连续相乘。Step 9.2 再研究它的数值规模。


## 3. 亲手做一次 3-Step Scalar BPTT

使用一个只有三个 scalar parameters 的 RNN：

$$a_t=w_xx_t+w_hh_{t-1}+b,\qquad h_t=\tanh(a_t).$$

取：

```text
w_x = 0.7
w_h = 0.5
b   = 0.1
x   = [1.0, 0.5, -0.5]
h_(-1) = 0
target = 0.2
```

只在最后状态定义：

$$L=\frac12(h_2-y)^2.$$

Forward 结果近似：

```text
a_0 = 0.800000   h_0 = 0.664037
a_1 = 0.782018   h_1 = 0.653864
a_2 = 0.076932   h_2 = 0.076780
L   = 0.00759153
```

Backward 从最后一步开始：

$$\frac{\partial L}{\partial h_2}=h_2-y\approx-0.12322.$$

定义

$$\delta_t\equiv\frac{\partial L}{\partial a_t}.$$

则

$$\delta_t=\frac{\partial L}{\partial h_t}(1-h_t^2).$$

于是：

```text
delta_2 ≈ -0.12249
dL/dh_1 = delta_2 * w_h ≈ -0.06125
delta_1 ≈ -0.03506
dL/dh_0 = delta_1 * w_h ≈ -0.01753
delta_0 ≈ -0.00980
```

这就是 BPTT：沿 recurrent graph 从晚到早逐步使用 chain rule。


In [ ]:
import math

wx = 0.7
wh = 0.5
bias = 0.1
xs = [1.0, 0.5, -0.5]
target = 0.2

h_prev = 0.0
hs = []
preacts = []
for x_t in xs:
    a_t = wx*x_t + wh*h_prev + bias
    h_t = math.tanh(a_t)
    preacts.append(a_t)
    hs.append(h_t)
    h_prev = h_t

loss = 0.5 * (hs[-1] - target)**2
dL_dh = hs[-1] - target
deltas = [0.0] * len(xs)

for t in reversed(range(len(xs))):
    delta_t = dL_dh * (1.0 - hs[t]**2)
    deltas[t] = delta_t
    dL_dh = delta_t * wh

print('pre-activations:', [round(v, 6) for v in preacts])
print('hidden states  :', [round(v, 6) for v in hs])
print('loss           :', round(loss, 8))
print('deltas dL/da_t :', [round(v, 6) for v in deltas])


Forward 是从过去到未来；reverse-mode backward 则从较晚节点向较早节点传播 sensitivity。

当前例子只有 final loss，所以早期 `h_t` 的 gradient 完全来自 future path。若每个时间步都有 local loss，还需加上第 2 节公式中的 local contribution。


## 4. Shared Parameter Gradient：同一个参数在多个时间步的贡献必须相加

由于

$$a_t=w_xx_t+w_hh_{t-1}+b,$$

有

$$\frac{\partial a_t}{\partial w_x}=x_t,\qquad
\frac{\partial a_t}{\partial w_h}=h_{t-1},\qquad
\frac{\partial a_t}{\partial b}=1.$$

同一个参数在所有 t 被重复使用，所以总 gradient 是各使用位置路径贡献的和：

$$
\boxed{\frac{\partial L}{\partial w_x}=\sum_t\delta_t x_t}
$$

$$
\boxed{\frac{\partial L}{\partial w_h}=\sum_t\delta_t h_{t-1}}
$$

$$
\boxed{\frac{\partial L}{\partial b}=\sum_t\delta_t}
$$

本例近似：

```text
dL/dw_x =  0.033915
dL/dw_h = -0.103376
dL/db   = -0.167355
```

向量 RNN 中结构类似，只是变为 outer-product sums：

$$
\frac{\partial L}{\partial W_x}=\sum_t\delta_t x_t^T
\in\mathbb R^{H\times C},
$$

$$
\frac{\partial L}{\partial W_h}=\sum_t\delta_t h_{t-1}^T
\in\mathbb R^{H\times H}.
$$

有 Batch 时还要合并 batch samples 的贡献。


In [ ]:
manual_dwx = 0.0
manual_dwh = 0.0
manual_db = 0.0

for t in range(len(xs)):
    h_before = 0.0 if t == 0 else hs[t-1]
    manual_dwx += deltas[t] * xs[t]
    manual_dwh += deltas[t] * h_before
    manual_db += deltas[t]

print('manual dL/dwx:', manual_dwx)
print('manual dL/dwh:', manual_dwh)
print('manual dL/db :', manual_db)


### Apple MLX Autodiff 验证

让 `mx.grad` 对同一个 recurrent forward 自动求导，结果应与手算一致。


In [ ]:
import mlx.core as mx

xs_mx = mx.array([1.0, 0.5, -0.5])
target_mx = mx.array(0.2)

def scalar_rnn_loss(theta):
    wx_mx = theta[0]
    wh_mx = theta[1]
    b_mx = theta[2]

    h = mx.array(0.0)
    for t in range(xs_mx.shape[0]):
        h = mx.tanh(wx_mx * xs_mx[t] + wh_mx * h + b_mx)

    return 0.5 * mx.square(h - target_mx)

theta = mx.array([0.7, 0.5, 0.1])
loss_mx = scalar_rnn_loss(theta)
grads_mx = mx.grad(scalar_rnn_loss)(theta)
manual_grads = mx.array([manual_dwx, manual_dwh, manual_db])
mx.eval(loss_mx, grads_mx, manual_grads)

print('MLX loss        :', loss_mx)
print('MLX gradients   :', grads_mx)
print('manual gradients:', manual_grads)
print('match:', bool(mx.allclose(grads_mx, manual_grads, atol=1e-5)))


MLX 不要求我们再手写一个特殊的“BPTT API”。只要 forward graph 中存在 recurrent dependencies，reverse-mode autodiff 就会沿展开图应用 chain rule，并把 shared parameter 的多条 gradient paths 汇总。


## 5. 长距离 Gradient 为什么会出现很多局部导数的乘积？

例如：

```text
h_0 -> h_1 -> h_2 -> h_3
```

Chain Rule 给出：

$$
\frac{\partial h_3}{\partial h_0}
=
\frac{\partial h_3}{\partial h_2}
\frac{\partial h_2}{\partial h_1}
\frac{\partial h_1}{\partial h_0}.
$$

推广：

$$
\boxed{
\frac{\partial h_T}{\partial h_t}
=
\prod_{k=t+1}^{T}
\frac{\partial h_k}{\partial h_{k-1}}
}
$$

scalar tanh RNN 中：

$$
\frac{\partial h_k}{\partial h_{k-1}}
=
w_h(1-h_k^2).
$$

所以跨很多时间步会出现许多这样的局部因子的乘积。

向量 RNN 中 scalar derivative 会升级为 Jacobian matrices 的连乘。当前只需要掌握这个结构。

这里不要提前下结论说一定变小或一定变大；实际取决于 weights、activation derivatives、状态和方向。Step 9.2 才正式分析 vanishing / exploding。

### BPTT 的资源代价预览

为了 backward 穿过时间，训练系统通常需要保留或重建 forward 的中间 states/activations，因此长 T 会增加 backward graph 长度和训练内存压力。

工程上存在 truncated BPTT、checkpointing 等方法。Truncated BPTT 会主动截断某些跨时间 gradient paths，是计算/记忆/长期 credit assignment 的 trade-off，不是与完整 BPTT 完全等价的免费优化。

### 9.1 最终结论

- BPTT 本质上是普通 reverse-mode backpropagation 应用于沿时间展开的 recurrent graph。
- 展开的多个 Cell 节点共享同一 `W_x/W_h/b`，不是多套模型参数。
- 中间 `h_t` 的 gradient 可以同时来自当前 local loss 与 future recurrent path。
- scalar RNN 可定义 `delta_t=dL/da_t` 并从晚到早递推。
- shared parameter 的总 gradient 是所有时间使用位置上的 path contributions 之和。
- MLX `mx.grad` 会自动沿 recurrent graph 完成 chain rule 并汇总 shared-parameter gradients。
- 长距离 dependency 的 gradient 包含多个局部 derivative/Jacobian 的乘积。
- 下一节 **9.2 — Vanishing and Exploding Gradients**：分析这些连续乘积的数值规模，以及 gradient clipping 能解决什么、不能解决什么。
